# Classificação de Roubos e Furtos de Celulares em São Paulo

**Global Solution FIAP 2026.2 · Notebook acadêmico consolidado · Modelo v1.0.0**

Uma jornada dos dados públicos à demonstração web: auditoria da fonte, exploração,
pré-processamento, primeira MLP, três arquiteturas, comparação controlada,
avaliação final congelada e revisão crítica independente.

**Como ler:** os resultados e gráficos já estão visíveis no GitHub. As células
executáveis padrão apenas leem agregados públicos e código-fonte. Não carregam
planilhas, boletins individuais, pesos, TensorFlow ou previsões privadas. Não
treinam nem avaliam novamente 2025. Os blocos opcionais de reprodução estão
desativados; não habilitá-los para consultar esta entrega.

Esta edição reorganiza os cinco notebooks anteriores, sem substituir as evidências
originais dos [relatórios](../reports/relatorio_etapa1.md). Sua rastreabilidade está
no [registro de consolidação](../docs/consolidacao_academica.md). Consolidação:
11/11/2026; resultados experimentais: 09/11/2026.

## Roteiro

1. [Introdução](#01-introducao)
2. [Base de dados](#02-base)
3. [Exploração e tratamento](#03-tratamento)
4. [Implementação da primeira MLP](#04-baseline)
5. [Experimentos com arquiteturas](#05-arquiteturas)
6. [Comparação dos modelos](#06-comparacao)
7. [Avaliação final](#07-final)
8. [Auditoria independente](#08-auditoria)
9. [Integração web](#09-web)
10. [Conclusões](#10-conclusoes)

<a id="01-introducao"></a>
## 01 — Introdução

O projeto aplica Deep Learning a um problema binário sobre **ocorrências já
registradas ou exemplos hipotéticos** envolvendo celulares subtraídos em São Paulo
capital. Dadas quatro características contextuais, a rede classifica o padrão
como **FURTO (0)** ou **ROUBO (1)**. A referência de rótulo é a rubrica da SSP-SP;
o modelo não determina a natureza jurídica de um novo fato.

O objetivo acadêmico é demonstrar o ciclo completo de uma MLP: preparação auditável,
controle de vazamento, implementação de pelo menos três arquiteturas, comparação
de métricas, curvas de aprendizado, matriz de confusão e discussão das limitações.
**Não se prevê onde, quando ou contra quem ocorrerá um crime.**

A fonte é a série [Celulares subtraídos da SSP-SP](https://www.ssp.sp.gov.br/estatistica/consultas).
O recorte é o município de São Paulo, fatos de 2023–2025. O material não representa
todos os crimes da cidade, nem uma estatística oficial produzida pelo projeto.

### Ambiente para leitura e reprodução

Para reexecutar as células de leitura, basta Python 3.12 com Jupyter/IPython:
`python -m pip install notebook nbformat nbclient ipykernel`.
Abra este arquivo na raiz ou em `notebooks/`. Para a reprodução histórica opcional,
são necessários `requirements-ml.txt`, os dados locais auditados e uma cópia
histórica separada. Para inferência web, os binários finais são necessários e
**não estão no GitHub**. Nenhum desses pré-requisitos pesados é necessário para ler
as saídas salvas abaixo.

In [1]:
from pathlib import Path
import csv, json, ast, hashlib
from html import escape
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'reports/etapa5/metricas_2025.json').is_file())
def ler_json(caminho):
    return json.loads((ROOT / caminho).read_text(encoding='utf-8'))
def ler_csv(caminho):
    with (ROOT / caminho).open(encoding='utf-8-sig', newline='') as arquivo:
        return list(csv.DictReader(arquivo))
def tabela(linhas, colunas):
    # Exibe apenas agregados e colunas explicitamente selecionadas.
    def valor(v): return str(v if v not in ('', None) else '—').replace('|', chr(92)+'|').replace('\n', ' ')
    assert all(set(colunas) <= set(linha) for linha in linhas), 'Coluna ausente no agregado'
    cabecalho = '| ' + ' | '.join(colunas) + ' |'
    separador = '| ' + ' | '.join('---' for _ in colunas) + ' |'
    corpo = ['| ' + ' | '.join(valor(linha.get(c, '')) for c in colunas) + ' |'
             for linha in linhas]
    display(Markdown('\n'.join([cabecalho, separador, *corpo])))
def percentual(v): return f'{float(v)*100:.2f}%'.replace('.', ',')
def mostrar_fonte(caminho, inicio, fim):
    from textwrap import dedent
    linhas = (ROOT/caminho).read_text(encoding='utf-8').splitlines()
    a = next(i for i, linha in enumerate(linhas) if inicio in linha)
    b = next(i for i, linha in enumerate(linhas[a:], a) if fim in linha)
    display(Markdown('```python\n' + dedent('\n'.join(linhas[a:b+1])) + '\n```'))
print('Leitura de agregados públicos; nenhum treinamento ou inferência de 2025.')

Leitura de agregados públicos; nenhum treinamento ou inferência de 2025.


<a id="02-base"></a>
## 02 — Base de dados

Os três arquivos XLSX originais foram fornecidos localmente, com metodologia e
dicionário da SSP-SP. A proveniência anterior ao fornecimento é declarada; não houve
comparação byte a byte com novos downloads. Os hashes e esquemas estão no
[inventário](../reports/inventario.json) e nas [decisões de origem](../docs/decisoes.md).
Cada arquivo tem 55 colunas e três abas. Em 2025, mês/ano do registro não devem
ser confundidos com a data do fato.

| Arquivo | Linhas originais | Tamanho decimal |
| --- | ---: | ---: |
| 2023 | 490.154 | 137,60 MB |
| 2024 | 412.527 | 118,61 MB |
| 2025 | 383.635 | 108,12 MB |

As **1.286.316 linhas** representam vínculos de **995.565 chaves distintas de BO**,
não 1,28 milhão de crimes independentes. Após seleção, são **505.187 boletins
elegíveis**, sustentados por 566.582 linhas de origem.

### Recorte, rótulos e duplicidades

O filtro principal usa `NOME_MUNICIPIO_CIRC`, município da circunscrição do fato;
`CD_IBGE=3550308` corrobora o recorte quando preenchido. Não se usa o município
do registro como local do fato, nem se afirma geocodificação independente.
O ano vem da **data do fato**, mesmo quando difere do ano do arquivo.

Os rótulos dependem de igualdade normalizada com `Furto (art. 155)` e
`Roubo (art. 157)`, sem busca por substring. Artigo 156 e outras naturezas não
são convertidos automaticamente. Todos os registros observados são consumados.

A unidade é o BO, consolidado por delegacia/ano/número. Não foram encontradas
linhas integralmente idênticas dentro dos arquivos; múltiplos vínculos, quatro
chaves entre arquivos, versões e naturezas exigiram consolidação. BOs com
conflitos ou outras naturezas foram segregados conservadoramente. Não é possível
garantir que dois BOs diferentes não descrevam o mesmo evento real.

In [2]:
tabela(ler_csv('reports/tables/decisoes_boletins.csv'),
       ['decisao', 'boletins', 'linhas_origem', 'pct_boletins'])
tabela(ler_csv('reports/tables/classes_total.csv'), ['alvo', 'boletins', 'percentual'])
tabela(ler_csv('reports/tables/classes_por_ano.csv'), ['ano_fato', 'alvo', 'boletins', 'pct_ano'])

| decisao | boletins | linhas_origem | pct_boletins |
| --- | --- | --- | --- |
| elegivel | 505187 | 566582 | 50.7437 |
| fora_capital_ou_indeterminado | 411798 | 570352 | 41.3632 |
| sem_classe_binaria | 61452 | 65158 | 6.1726 |
| outras_naturezas_associadas | 12708 | 64564 | 1.2765 |
| fato_fora_2023_2025 | 2879 | 3311 | 0.2892 |
| geografia_conflitante | 1392 | 15500 | 0.1398 |
| roubo_e_furto_no_bo | 149 | 849 | 0.015 |

| alvo | boletins | percentual |
| --- | --- | --- |
| 0 | 298083 | 59.0045 |
| 1 | 207104 | 40.9955 |

| ano_fato | alvo | boletins | pct_ano |
| --- | --- | --- | --- |
| 2023 | 0 | 106421 | 55.9745 |
| 2023 | 1 | 83703 | 44.0255 |
| 2024 | 0 | 92055 | 57.9345 |
| 2024 | 1 | 66840 | 42.0655 |
| 2025 | 0 | 99607 | 63.782 |
| 2025 | 1 | 56561 | 36.218 |

As decisões são mutuamente exclusivas por prioridade. **490.378 BOs segregados**
não equivalem às **719.734 linhas fora dos BOs elegíveis**; outras 61.395 linhas
são vínculos consolidados dos BOs mantidos. Dos elegíveis, 298.083 são furtos
(59,00%) e 207.104 roubos (41,00%). Não houve balanceamento artificial.

A exclusão conservadora reduz a ambiguidade jurídica, mas seleciona casos mais
simples: a auditoria encontrou exclusão de outras naturezas em 4,3% dos roubos
e 1,1% dos furtos. Essa limitação não foi corrigida alterando a base histórica.

A Etapa 1 também preparou amostra opcional de 20 mil BOs, com cotas proporcionais
por ano/mês/classe pelo método de maior resto e ordenação SHA256 determinística.
Ela é subconjunto da base ampliada, não avaliação independente. Os treinamentos
usaram os conjuntos completos permitidos, sem depender dessa amostra.

<a id="03-tratamento"></a>
## 03 — Exploração e tratamento

`NULL`, vazio, `N/A` e `NAN` foram tratados como ausência. A padronização preserva
a fonte e usa trim/maiúsculas nas comparações necessárias. Há 184 BOs com conflito
de local e 470 com conflito de bairro, convertidos em ausente por consenso,
sem escolher arbitrariamente a primeira linha. Datas inconsistentes foram
diagnosticadas; não se inventaram datas ou horas para corrigi-las.

O perfil abaixo descreve os cinco candidatos originais. **`periodo_fato` foi
excluído da MLP**: ausência complementar de hora/período e sua associação à classe
podem refletir o modo de registro. As faixas exploratórias do período eram uma
convenção do projeto (00–05, 06–11, 12–17, 18–23), não definição oficial da SSP.

In [3]:
tabela(ler_json('reports/perfil_atributos.json'),
       ['atributo', 'linhas', 'ausentes', 'ausentes_pct', 'cardinalidade'])

| atributo | linhas | ausentes | ausentes_pct | cardinalidade |
| --- | --- | --- | --- | --- |
| mes | 505187 | 0 | 0.0 | 12 |
| dia_semana | 505187 | 0 | 0.0 | 7 |
| periodo_fato | 505187 | 0 | 0.0 | 5 |
| tipo_local | 505187 | 18752 | 3.7119 | 28 |
| bairro | 505187 | 1691 | 0.3347 | 2328 |

![Exploração original: distribuição de classes, tempo e qualidade](../reports/figures/painel_exploratorio.png)

*Exploração original: distribuição de classes, tempo e qualidade. Figura histórica preservada; não é regenerada nesta consolidação.*

### Representação final e prevenção de vazamento

Entradas exatas: **mês, dia da semana, tipo de local e bairro**. Mês 1–12 e dia
0–6 (segunda=0) são categóricos nominais. Natureza, lei, identificadores, endereço,
coordenadas precisas, ano, data completa e indicadores administrativos não entram
na rede. Chaves e datas ficam em metadados para auditar alinhamento e disjunção.

O pré-processador usa Unicode NFC, espaços/maiúsculas e somente aliases ativos
aprendidos/confirmados em 2023. Não funde globalmente bairros com ou sem acento,
nem abreviações JD/VL. Na versão final, equivalências unívocas de acentuação de
**local** foram adicionadas na Etapa 4; `OUTROS` continua distinto de ausente.

Bairros com menos de **50 exemplos no treino** compartilham um bucket estatístico.
São 247 bairros com coluna própria dentre 1.913 nomes conhecidos em 2023. O bucket
não afirma equivalência geográfica entre bairros. Ausências têm categoria explícita;
não se removem exemplos. Não há escalonamento ou imputação por moda.

One-Hot resulta em **299 colunas = 13 de mês + 8 de dia + 29 de local + 249 de bairro**.
A matriz é CSR float32; somente lotes de até 256 linhas são densificados.
O pipeline histórico tem fallback de desconhecidos, mas o contrato seguro da web
**rejeita categorias desconhecidas** e obriga mês/dia; não confundir as duas interfaces.

### Separação temporal

| Papel | Período | BOs | Uso |
| --- | --- | ---: | --- |
| Treino | 2023 | 190.124 | Ajuste dos transformadores e da rede |
| Desenvolvimento | 2024 | 158.895 | Validação histórica, diagnóstico e seleção |
| Avaliação final histórica | 2025 | 156.168 | Teste consumido; resultados somente lidos |

Na Etapa 4, janeiro–setembro/2023 (144.697) treina e outubro–dezembro/2023 (45.427)
orienta escolhas internas. Depois, todo 2023 é reajustado por três épocas fixas,
e 2024 é avaliado apenas ao final. Os meses Q4 não aparecem no treino interno:
seus dummies não aprendem efeitos, limitando as conclusões sobre o atributo mês.

Vocabulários, raridade e aliases são ajustados somente no treino correspondente;
transformar validação não modifica o objeto. Chaves de BO são únicas/disjuntas.
Isso evita vazamento conhecido, sem provar ausência universal de proxies.
**2025 já teve estatísticas exploradas na Etapa 1 e foi consultado na auditoria;
não é um teste cego disponível para uma nova versão.**

In [4]:
tabela(ler_json('reports/etapa4/divisao_interna.json'),
       ['particao', 'n', 'mes_min', 'mes_max', 'furtos', 'roubos'])

| particao | n | mes_min | mes_max | furtos | roubos |
| --- | --- | --- | --- | --- | --- |
| treino_interno | 144697 | 1 | 9 | 81128 | 63569 |
| validacao_interna | 45427 | 10 | 12 | 25293 | 20134 |

<a id="04-baseline"></a>
## 04 — Implementação da primeira MLP

O baseline histórico da Etapa 2 já usava **299 → Dense64 ReLU → 1 sigmoid**,
19.265 parâmetros. **Não é o mesmo checkpoint da Dense64 final.** Treinou em 2023
com BCE, Adam 0,001, batch 256, semente 42, máximo 30 épocas e Early Stopping em
`val_loss` de **2024**, paciência 5, `min_delta=0`, restauração dos melhores pesos.
Foram 10 épocas; os pesos restaurados são os da época 5. Não houve ajuste em 2025.

O código a seguir é extraído do treinamento original. A célula apenas apresenta
o fonte; **não importa nem chama `ml.train`**.

In [5]:
mostrar_fonte('src/ml/train.py', 'model=tf.keras.Sequential',
              'callbacks=[early,LogEpoch()],shuffle=False,verbose=0)')

```python
model=tf.keras.Sequential([tf.keras.layers.Input(shape=(dimension,),name='atributos_one_hot'),
  tf.keras.layers.Dense(64,activation='relu',name='oculta_relu'),tf.keras.layers.Dense(1,activation='sigmoid',name='probabilidade_roubo')])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),loss='binary_crossentropy',metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])
epochs=[];early=tf.keras.callbacks.EarlyStopping(monitor='val_loss',mode='min',patience=5,min_delta=0,restore_best_weights=True)
training_start=time.perf_counter()
model.fit(Batches(X,y,True),validation_data=Batches(V,vy,False),epochs=30,
          callbacks=[early,LogEpoch()],shuffle=False,verbose=0)
```

O treinamento real é `ml.train.train(min_bairro_count=50)` em
[src/ml/train.py](../src/ml/train.py), com `Batches` densificando somente cada lote.
O [guia histórico](../docs/etapa2_reproducao.md) descreve os dados e dependências.
As chamadas de reprodução opcional aparecem juntas na seção 05, para não dispersar
comandos que possam sobrescrever artefatos.

In [6]:
baseline = ler_csv('reports/tables/metricas_baseline.csv')
tabela(baseline, ['modelo', 'ano', 'n', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'balanced_accuracy'])
historico = ler_json('reports/etapa2/historico_mlp.json')
tabela([{'epoca': r['epoca'], 'loss': round(r['loss'], 6), 'val_loss': round(r['val_loss'], 6),
         'accuracy': percentual(r['accuracy']), 'val_accuracy': percentual(r['val_accuracy'])}
        for r in historico], ['epoca', 'loss', 'val_loss', 'accuracy', 'val_accuracy'])

| modelo | ano | n | accuracy | precision | recall | f1 | macro_f1 | balanced_accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MLP | 2023 | 190124 | 0.7014527361090657 | 0.6778349834983498 | 0.6134308208785826 | 0.6440267665079992 | 0.6934755993367914 | 0.692057589144622 |
| MLP | 2024 | 158895 | 0.6872840555083546 | 0.6327374042256791 | 0.6115798922800718 | 0.6219787743923314 | 0.6776641034482787 | 0.6769159034481669 |
| Majoritario | 2024 | 158895 | 0.5793448503728877 | 0.0 | 0.0 | 0.0 | 0.3668260609683204 | 0.5 |

| epoca | loss | val_loss | accuracy | val_accuracy |
| --- | --- | --- | --- | --- |
| 1 | 0.596177 | 0.588378 | 67,95% | 68,68% |
| 2 | 0.582458 | 0.58552 | 69,15% | 68,73% |
| 3 | 0.578996 | 0.586569 | 69,38% | 68,51% |
| 4 | 0.5762 | 0.58518 | 69,68% | 68,65% |
| 5 | 0.57411 | 0.584672 | 69,79% | 68,73% |
| 6 | 0.57222 | 0.58551 | 69,89% | 68,79% |
| 7 | 0.570762 | 0.588292 | 70,06% | 68,27% |
| 8 | 0.569427 | 0.587569 | 70,19% | 68,49% |
| 9 | 0.568332 | 0.587143 | 70,28% | 68,65% |
| 10 | 0.567224 | 0.587108 | 70,30% | 68,53% |

![Baseline: curvas reais de treino 2023 e validação 2024](../reports/figures/curvas_mlp_baseline.png)

*Baseline: curvas reais de treino 2023 e validação 2024. Figura histórica preservada; não é regenerada nesta consolidação.*

![Baseline: matriz de confusão em 2024](../reports/figures/matriz_confusao_baseline.png)

*Baseline: matriz de confusão em 2024. Figura histórica preservada; não é regenerada nesta consolidação.*

### Diagnóstico intermediário — Etapa 3

A matriz histórica de 2024 é TN=68.328, FP=23.727, FN=25.962, TP=40.878.
Após a época 5, loss cai enquanto val_loss piora: é compatível com sobreajuste
incremental, mas o gap também reúne mudança temporal. Não demonstra, sozinho,
underfitting ou necessidade de mais camadas.

Permutação dos atributos originais em amostra fixa de 30 mil BOs de 2024, cinco
repetições, indicou maior dependência de bairro e local. Permutação marginal quebra
dependências; não mede causalidade e seu DP não é IC populacional.
Nos raros conhecidos de 2024, recall de roubo ≈95,32% coexistiu com FPR ≈75,23%.
Sem local, todos os 4.653 exemplos foram classificados como furto, incluindo
479 roubos perdidos. Alto recall ou acurácia isolados podem esconder falhas.

In [7]:
tabela(ler_csv('reports/etapa3/importancia_atributos.csv'),
       ['atributo', 'delta_macro_f1_media', 'delta_accuracy_media', 'delta_macro_f1_dp'])
tabela(ler_csv('reports/etapa3/estabilidade_sementes.csv'),
       ['seed', 'epochs', 'best_epoch', 'accuracy', 'macro_f1', 'recall_roubo'])

| atributo | delta_macro_f1_media | delta_accuracy_media | delta_macro_f1_dp |
| --- | --- | --- | --- |
| mes | 0.00012156612196563276 | 0.00043333333333333 | 0.0011696679743555998 |
| dia_semana | 0.0022133274689665862 | 0.0028799999999999937 | 0.0011016497790114251 |
| tipo_local | 0.06472890262016365 | 0.05888000000000002 | 0.0018378391766285932 |
| bairro | 0.11664847776174299 | 0.10946666666666667 | 0.003197507886261515 |

| seed | epochs | best_epoch | accuracy | macro_f1 | recall_roubo |
| --- | --- | --- | --- | --- | --- |
| 42 | 10 | 5 | 0.6872840555083546 | 0.6776641034482787 | 0.6115798922800718 |
| 7 | 10 | 5 | 0.6869442084395355 | 0.6771218303984325 | 0.6092010771992818 |
| 123 | 10 | 5 | 0.687372164007678 | 0.6777346873765661 | 0.6114751645721125 |

![Etapa 3: importância marginal por permutação, não causal](../reports/etapa3/figures/importancia_atributos.png)

*Etapa 3: importância marginal por permutação, não causal. Figura histórica preservada; não é regenerada nesta consolidação.*

In [8]:
tabela(ler_csv('reports/etapa3/drift_atributos.csv'),
       ['atributo', 'distancia_variacao_total', 'categorias_2023', 'categorias_2024', 'registros_novos_2024'])
grupos = [r for r in ler_csv('reports/etapa3/metricas_grupos.csv')
          if r['atributo'] in ['condicao_bairro', 'presenca_local']]
tabela(grupos, ['atributo', 'categoria', 'n', 'furtos', 'roubos', 'macro_f1', 'recall_roubo', 'fpr'])

| atributo | distancia_variacao_total | categorias_2023 | categorias_2024 | registros_novos_2024 |
| --- | --- | --- | --- | --- |
| mes | 0.01812989892246382 | 12 | 12 | 0 |
| dia_semana | 0.007601141166298925 | 7 | 7 | 0 |
| tipo_local | 0.057568487175362525 | 29 | 29 | 0 |
| bairro | 0.08735976532966672 | 1914 | 1812 | 355 |

| atributo | categoria | n | furtos | roubos | macro_f1 | recall_roubo | fpr |
| --- | --- | --- | --- | --- | --- | --- | --- |
| condicao_bairro | ausente | 521 | 167 | 354 | 0.6331075010320293 | 0.8926553672316384 | 0.6407185628742516 |
| condicao_bairro | desconhecido | 355 | 130 | 225 | 0.6153036854674957 | 0.9511111111111111 | 0.7076923076923077 |
| condicao_bairro | frequente | 145346 | 86672 | 58674 | 0.6707423134299194 | 0.5644067218870369 | 0.22731678050581502 |
| condicao_bairro | raro_conhecido | 12673 | 5086 | 7587 | 0.575910424947882 | 0.9532094371952023 | 0.7522611089264648 |
| presenca_local | ausente | 4653 | 4174 | 479 | 0.4728673388467203 | 0.0 | 0.0 |
| presenca_local | preenchido | 154242 | 87881 | 66361 | 0.673521149014509 | 0.6159943340214885 | 0.2699901002492006 |

![Baseline: confiabilidade em 2024, sem calibrador](../reports/etapa3/figures/curva_calibracao.png)

*Baseline: confiabilidade em 2024, sem calibrador. Figura histórica preservada; não é regenerada nesta consolidação.*

![Baseline: limiares diagnósticos; nenhum substituiu o limiar 0,5](../reports/etapa3/figures/precision_recall_limiares.png)

*Baseline: limiares diagnósticos; nenhum substituiu o limiar 0,5. Figura histórica preservada; não é regenerada nesta consolidação.*

O controle semente 42 reproduziu o baseline; sementes 7 e 123 sugeriram baixa
variação de Macro F1, mas com apenas três inicializações e lotes sempre RNG42.
Confiança ≥90% apareceu em 879 erros (865 FN). Brier=0,200906,
Log Loss=0,584672, ECE10=0,020581 são diagnósticos **do baseline**, não do final.
Sondagens encontraram mês/dia ausentes sem exemplos de treino e fallbacks inseguros:
foram documentados nessa etapa e tratados externamente após a auditoria.

As tabelas completas de drift, ambiguidade de contextos, ausências, confiança,
limiares diagnósticos e robustez estão no [relatório 3](../reports/relatorio_etapa3.md).
O teto empírico por contexto no próprio conjunto não é limite Bayesiano nem
desempenho alcançável em dados novos. Nenhum limiar foi escolhido nessas análises.

A recarga histórica do baseline reproduziu as previsões de 2024 com diferença
máxima zero. No exemplo sintético de maio/quarta/VIA PÚBLICA/BELA VISTA,
o baseline retornou FURTO e score de roubo 0,22824792563915253, enquanto o checkpoint
final retorna 0,25535914301872253. São pesos/protocolos diferentes, não divergência
de recarga. O exemplo antigo é transcrito do relatório, sem inferência nova.

<a id="05-arquiteturas"></a>
## 05 — Experimentos com arquiteturas

A Etapa 4 implementou e treinou **Dense32, Dense64 e Dense64 → Dense32**.
O experimento investiga capacidade/profundidade na **mesma representação**, sem
alterar simultaneamente atributos ou orçamento. ReLU nas ocultas, sigmoid na saída,
BCE, Adam 0,001 e lote 256 são comuns. No escopo interno há máximo 30 épocas,
Early Stopping em Q4/2023 e paciência 5. No reajuste de 2023 completo são três
épocas fixas, sem monitorar 2024 por época.

O trecho executável abaixo **somente lê** a implementação efetivamente usada em
[Engine.run](../src/experiments4/engine.py). Inclui construção, compile,
lotes esparsos e `model.fit`, em vez de apresentar apenas tabelas prontas.
`spec['units']` recebe `[32]`, `[64]` ou `[64, 32]` no
[orquestrador real](../src/experiments4/run.py).

In [9]:
mostrar_fonte('src/experiments4/engine.py',
              'tf=self.tf;tf.keras.backend.clear_session()',
              'train_seconds=time.perf_counter()-fit_start;history=')

```python
tf=self.tf;tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(seed)
layers=[tf.keras.layers.Input(shape=(X.shape[1],))]
for units in spec['units']:
    layers.append(tf.keras.layers.Dense(units,activation='relu',kernel_regularizer=tf.keras.regularizers.L2(spec['l2']) if spec['l2'] else None))
    if spec['dropout']:layers.append(tf.keras.layers.Dropout(spec['dropout']))
layers.append(tf.keras.layers.Dense(1,activation='sigmoid'));model=tf.keras.Sequential(layers)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=spec['lr']),loss='binary_crossentropy',metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])
class Batches(tf.keras.utils.PyDataset):
    def __init__(self,x,labels,shuffle):
        super().__init__(workers=1,use_multiprocessing=False,max_queue_size=2)
        self.x=x;self.y=labels;self.shuffle=shuffle;self.ids=np.arange(len(labels));self.rng=np.random.default_rng(42)
        if shuffle:self.rng.shuffle(self.ids)
    def __len__(self):return (len(self.ids)+255)//256
    def __getitem__(self,i):
        ids=self.ids[i*256:(i+1)*256];return self.x[ids].toarray(),self.y[ids].astype(np.float32).reshape(-1,1)
    def on_epoch_end(self):
        if self.shuffle:self.rng.shuffle(self.ids)
class Progress(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        if epoch==0 or (epoch+1)%5==0:print(f'{path.stem}: epoca {epoch+1}, loss {logs["loss"]:.5f}'+(f', val_loss {logs["val_loss"]:.5f}' if 'val_loss' in logs else ''),flush=True)
early=tf.keras.callbacks.EarlyStopping(monitor='val_loss',mode='min',patience=5,restore_best_weights=True)
fit_start=time.perf_counter();h=model.fit(Batches(X,y,True),validation_data=Batches(V,vy,False) if scope=='internal' else None,epochs=30 if scope=='internal' else epochs,
    callbacks=[early,Progress()] if scope=='internal' else [Progress()],shuffle=False,verbose=0)
train_seconds=time.perf_counter()-fit_start;history={k:[float(v) for v in values] for k,values in h.history.items()}
```

### Reprodução opcional — não executada nesta entrega

Use uma **cópia histórica isolada** da tag `snapshot-pos-auditoria-antes-correcoes`,
com os Parquet auditados, dependências ML e artefatos históricos necessários aos
controles de preservação. Exemplo de criação: `git worktree add --detach ../GS-MLP-historico
snapshot-pos-auditoria-antes-correcoes`. A tag contém os cinco notebooks e fontes
originais; não é atestação retroativa do congelamento. Consulte os relatórios para
os hashes dos arquivos locais, ausentes no clone. O pipeline de dados é
`python src/data/run.py --input-dir <pasta-dos-XLSX>`, apenas em reprodução separada.
Não é necessário executá-lo para visualizar esta entrega.

Os scripts históricos verificam bytes preservados: não os rode no checkout
reorganizado. Os comandos abaixo ficam bloqueados por padrão e usam a cópia
histórica explicitamente indicada, nunca `models/final` deste checkout.
Chamadas reais: `ml.train.train`, `experiments4.run.block_a`, `block_bc` e `refit`.
`block_bc` chama `Engine.run` para as três arquiteturas e regularizações previstas;
o cache só reutiliza execuções com configuração e hashes válidos. Se faltarem
artefatos de cache, a reprodução histórica exige preparação própria; não apague
registros para forçar treino. **Não há chamada a final5.evaluate ou seleção em 2025.**

In [10]:
# Todas as células podem ser executadas sem iniciar treinamento.
REPRODUZIR_BASELINE = False
REPRODUZIR_ETAPA4 = False
COPIA_HISTORICA = None  # Caminho da worktree histórica preparada manualmente.

if REPRODUZIR_BASELINE or REPRODUZIR_ETAPA4:
    import subprocess, sys, os
    assert COPIA_HISTORICA is not None, 'Prepare uma cópia histórica separada.'
    copia = Path(COPIA_HISTORICA).resolve()
    assert copia != ROOT.resolve(), 'Não escrever nos artefatos desta entrega.'
    tag = subprocess.check_output(['git', 'rev-parse', 'snapshot-pos-auditoria-antes-correcoes'],
                                  cwd=ROOT, text=True).strip()
    atual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=copia, text=True).strip()
    assert atual == tag, 'Use a versão histórica indicada para os controles de preservação.'
    ambiente = dict(os.environ, PYTHONPATH=str(copia/'src'), PYTHONHASHSEED='42')
    comandos = []
    if REPRODUZIR_BASELINE:
        comandos.append([sys.executable, '-m', 'ml.train'])
    if REPRODUZIR_ETAPA4:
        comandos += [[sys.executable, '-m', 'experiments4.plan']]
        comandos += [[sys.executable, '-m', 'experiments4.run', bloco]
                     for bloco in ['A', 'BC', 'refit']]
    for comando in comandos:
        subprocess.run(comando, cwd=copia, env=ambiente, check=True)
else:
    print('Reprodução desativada: nenhum modelo construído, treinado ou sobrescrito.')

Reprodução desativada: nenhum modelo construído, treinado ou sobrescrito.


### Hipóteses e controle experimental

**Bloco A:** retirar local/bairro/mês isoladamente; comparar frequência mínima
10/50/100 para bairros; testar equivalência unívoca de acento de local (A5).
A5 e A0 produziram matrizes idênticas; houve reutilização de pesos, sem ganho
métrico alegado. A regra mecânica de desempate apontou A3 (sem mês); houve revisão
qualitativa documentada antes do Bloco B para preservar recall. Não se apresenta
essa decisão como seleção totalmente automática pré-registrada.

**Bloco B:** comparar as três capacidades, com representação A5 fixa.
**Bloco C:** dropout 0,1 e L2=0,0001, separadamente. Nenhum melhorou Macro F1
interno. L2 reduziu gap, sem melhorar BCE de validação; loss penalizado não é
BCE pura. O gatilho prévio de oscilação para LR 0,0005 não ocorreu, então não foi
testado. Dense128→64 não foi implementada nesta bateria.

Foram 25 registros: 23 treinamentos novos e duas reutilizações comprovadas.
B64 e B64→32 receberam sementes globais 42/7/123; ordem dos lotes permaneceu RNG42.
Antes de comparar em 2024, representação, candidatos e orçamento foram congelados
na [agenda](../reports/etapa4/agenda_reajuste_congelada.json).

In [11]:
tabela(ler_csv('reports/etapa4/comparacao_atributos.csv'),
       ['id', 'input_dimension', 'retained_bairros', 'macro_f1', 'recall_roubo', 'delta_macro_f1_pp'])
tabela(ler_csv('reports/etapa4/comparacao_regularizacao.csv'),
       ['id', 'macro_f1', 'recall_roubo', 'log_loss', 'gap_bce'])

| id | input_dimension | retained_bairros | macro_f1 | recall_roubo | delta_macro_f1_pp |
| --- | --- | --- | --- | --- | --- |
| A0 | 267 | 215 | 0.6808291307405228 | 0.645226979239098 | 0.0 |
| A1 | 238 | 215 | 0.6477646199294143 | 0.5734578325221019 | -3.3064510811108505 |
| A2 | 50 | 0 | 0.5786361424736235 | 0.7258865600476806 | -10.219298826689926 |
| A3 | 254 | 215 | 0.6821647478740092 | 0.6064368729512268 | 0.13356171334863998 |
| A4_100 | 205 | 153 | 0.6794270996543571 | 0.6342505215059104 | -0.1402031086165656 |
| A4_10 | 671 | 619 | 0.6813945255308467 | 0.623870070527466 | 0.056539479032391426 |
| A5 | 267 | 215 | 0.6808291307405228 | 0.645226979239098 | 0.0 |

| id | macro_f1 | recall_roubo | log_loss | gap_bce |
| --- | --- | --- | --- | --- |
| B64 | 0.6808291307405228 | 0.645226979239098 | 0.5870552784758206 | 0.01064434240685952 |
| C_dropout | 0.6806834144918027 | 0.6680242376080262 | 0.5890381244580806 | 0.017378339893530437 |
| C_l2 | 0.6796415446787711 | 0.6660872156551108 | 0.5875692540975926 | 0.008573045280406033 |

![Etapa 4: ablações internas e sensibilidade da representação](../reports/etapa4/figuras/comparacao_atributos.png)

*Etapa 4: ablações internas e sensibilidade da representação. Figura histórica preservada; não é regenerada nesta consolidação.*

<a id="06-comparacao"></a>
## 06 — Comparação dos modelos

A tabela abaixo é a comparação real da **Etapa 4 em 2024, semente 42**, após
reajustar 2023 completo por três épocas fixas. Não mistura o baseline histórico
de Early Stopping em 2024. Precision, recall e F1 da tabela usam ROUBO como classe
positiva. Macro F1 é a média dos F1 das duas classes; balanced accuracy é a média
dos recalls por classe. Accuracy é a fração de acertos, sensível à prevalência.

In [12]:
comparacao = ler_csv('reports/etapa4/comparacao_arquiteturas.csv')
colunas_metricas = ['accuracy', 'precision_roubo', 'recall_roubo', 'f1_roubo', 'macro_f1', 'balanced_accuracy']
resumo = [{'Modelo': r['id'], 'Parâmetros': r['parameters'], 'Épocas': r['epochs'],
           **{k: percentual(r[k]) for k in colunas_metricas}} for r in comparacao]
tabela(resumo, ['Modelo', 'Parâmetros', 'Épocas', *colunas_metricas])

| Modelo | Parâmetros | Épocas | accuracy | precision_roubo | recall_roubo | f1_roubo | macro_f1 | balanced_accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| B32 | 9633 | 3 | 68,60% | 62,80% | 62,18% | 62,49% | 67,75% | 67,72% |
| B64_32 | 21313 | 3 | 68,10% | 61,43% | 64,98% | 63,15% | 67,52% | 67,68% |
| B64 | 19265 | 3 | 68,51% | 62,58% | 62,53% | 62,55% | 67,69% | 67,69% |

![Três arquiteturas: métricas em 2024 e custo em parâmetros](../reports/etapa4/figuras/comparacao_arquiteturas.png)

*Três arquiteturas: métricas em 2024 e custo em parâmetros. Figura histórica preservada; não é regenerada nesta consolidação.*

![Curvas reais das três MLPs: treino Jan–Set/2023 e validação Q4/2023](../reports/etapa4/figuras/curvas_arquiteturas.png)

*Curvas reais das três MLPs: treino Jan–Set/2023 e validação Q4/2023. Figura histórica preservada; não é regenerada nesta consolidação.*

![Reajuste em todo 2023: curvas somente de treino, sem validação por época em 2024](../reports/etapa4/figuras/curvas_reajuste.png)

*Reajuste em todo 2023: curvas somente de treino, sem validação por época em 2024. Figura histórica preservada; não é regenerada nesta consolidação.*

![Matrizes de confusão dos modelos reajustados em 2024](../reports/etapa4/figuras/matrizes_2024.png)

*Matrizes de confusão dos modelos reajustados em 2024. Figura histórica preservada; não é regenerada nesta consolidação.*

In [13]:
tabela(ler_csv('reports/etapa4/resumo_validacao_2024.csv'),
       ['id', 'n_seeds', 'parameters', 'macro_f1_mean', 'macro_f1_sd', 'recall_roubo_mean'])

| id | n_seeds | parameters | macro_f1_mean | macro_f1_sd | recall_roubo_mean |
| --- | --- | --- | --- | --- | --- |
| A0 | 1 | 19265 | 0.676902006111324 | — | 0.6253141831238779 |
| A1 | 1 | 17409 | 0.643565240944735 | — | 0.5616247755834829 |
| B32 | 1 | 9633 | 0.677451837885561 | — | 0.6218282465589468 |
| B64 | 3 | 19265 | 0.6770998373293823 | 0.00018284214421734156 | 0.6268950728106922 |
| B64_32 | 3 | 21313 | 0.6759000870303581 | 0.0006811888617745396 | 0.6485138639537203 |

![Estabilidade em três inicializações, com ordem dos lotes fixa](../reports/etapa4/figuras/estabilidade_sementes.png)

*Estabilidade em três inicializações, com ordem dos lotes fixa. Figura histórica preservada; não é regenerada nesta consolidação.*

### Por que Dense64?

Dense64 apresentou Macro F1 médio 0,67710 (DP 0,00018) nas três sementes completas;
a rede profunda teve 0,67590 (DP 0,00068), com maior recall de roubo e mais falsos
positivos. Dense32 obteve 0,67745 na semente 42 e usa 9.633 parâmetros, mas só teve
uma semente de reajuste completo. **Dense32 não foi demonstrada inferior**.

A seleção histórica, definida pelo responsável antes da avaliação final, escolheu
**B64_full_s42** pela estabilidade examinada, simplicidade frente à profunda e
ausência de ganho consistente de capacidade adicional. Não escolheu a melhor
semente por 2025. Não há evidência de que a Dense64 supere todas as alternativas.
Comparar com o baseline histórico (Macro F1 0,67766) exige cautela: ele usou 2024
para restaurar a época 5, enquanto o protocolo novo fixou três épocas internamente.

O maior recall da profunda troca FN por FP. Menor gap ou menos erros muito
confiantes não demonstra, por si só, melhor generalização ou calibração.
Conclusões e experimentos inferiores permanecem no [relatório 4](../reports/relatorio_etapa4.md).

<a id="07-final"></a>
## 07 — Avaliação final

**Leitura de evidência congelada: nenhuma nova inferência ou métrica de 2025.**
O artefato B64_full_s42 foi copiado sem ajuste para `models/final`. Modelo v1.0.0,
299→64 ReLU→1 sigmoid, três épocas, seed 42, limiar **0,5**, sem dropout/L2 e sem
calibrador. O manifesto registra hashes de pesos, pré-processador, configuração,
contrato e oito fontes de inferência. Não mover a classe serializada
`experiments4.preprocessing.ExperimentalPreprocessor`.

O congelamento foi registrado em 09/11/2026, 19:42:06 de São Paulo. A avaliação
planejada de 2025 ocorreu depois, sobre 156.168 BOs. A auditoria reproduziu e
diagnosticou esse teste posteriormente; **não se afirma que o teste permaneceu
cego em toda a história do projeto**. O Git foi iniciado após a auditoria.

In [14]:
manifesto = ler_json('models/final/manifesto.json')
final = ler_json('reports/etapa5/metricas_2025.json')
validacao_final = ler_json('reports/etapa5/metricas_2024_pipeline_final.json')
print('Modelo:', manifesto['model_id'])
print('Arquitetura:', manifesto['architecture'])
print('Limiar:', manifesto['threshold'], '| Calibrador adotado:', manifesto['calibration']['adopted'])
tabela([{'Métrica': k, '2025': percentual(final[k])} for k in
        ['accuracy', 'macro_f1', 'balanced_accuracy', 'precision_roubo', 'recall_roubo', 'f1_roubo']],
       ['Métrica', '2025'])
por_classe = [{'Classe': nome, 'Suporte': final[suporte],
               **{k: percentual(final[f'{k}_{prefixo}']) for k in ['precision', 'recall', 'f1']}}
              for nome, prefixo, suporte in [('FURTO', 'furto', 'furtos'), ('ROUBO', 'roubo', 'roubos')]]
tabela(por_classe, ['Classe', 'Suporte', 'precision', 'recall', 'f1'])

Modelo: gs-celulares-sp-dense64-v1.0.0
Arquitetura: {'input': 299, 'hidden': [64], 'activations': ['relu', 'sigmoid'], 'parameters': 19265}
Limiar: 0.5 | Calibrador adotado: False


| Métrica | 2025 |
| --- | --- |
| accuracy | 68,82% |
| macro_f1 | 67,09% |
| balanced_accuracy | 67,65% |
| precision_roubo | 56,15% |
| recall_roubo | 63,42% |
| f1_roubo | 59,56% |

| Classe | Suporte | precision | recall | f1 |
| --- | --- | --- | --- | --- |
| FURTO | 99607 | 77,58% | 71,88% | 74,62% |
| ROUBO | 56561 | 56,15% | 63,42% | 59,56% |

![Matriz final histórica de 2025 e referência majoritária](../reports/etapa5/matriz_confusao_2025.png)

*Matriz final histórica de 2025 e referência majoritária. Figura histórica preservada; não é regenerada nesta consolidação.*

O título original da figura usa “independente”; a ressalva pós-auditoria acima
qualifica essa expressão. A figura original foi preservada, sem editar a evidência.

Linhas = classe verdadeira; colunas = classe prevista; ordem FURTO, ROUBO.
**TN=71.598, FP=28.009, FN=20.691, TP=35.870**. São 48.700 erros; não confundir
falsos positivos entre furtos com fração de erros totais. A referência majoritária
usa FURTO, definido pelo treino, com Accuracy 63,78% e recall de roubo zero.

Os IC95% históricos usam 10 mil réplicas IID da matriz conjunta (y,pred), seed12345.
São condicionais ao modelo e ignoram dependência temporal/variabilidade de treino.
A célula abaixo somente lê os intervalos salvos; não faz bootstrap novamente.

In [15]:
intervalos = ler_json('reports/etapa5/intervalos_confianca_2025.json')['metrics']
tabela([{'Métrica': k, 'Estimativa': percentual(v['point']),
         'IC95% inferior': percentual(v['lower']), 'IC95% superior': percentual(v['upper'])}
        for k, v in intervalos.items()], ['Métrica', 'Estimativa', 'IC95% inferior', 'IC95% superior'])
tabela([{'Métrica': k, '2024': percentual(validacao_final[k]), '2025': percentual(final[k])}
        for k in ['accuracy', 'macro_f1', 'precision_roubo', 'recall_roubo', 'ece10']],
       ['Métrica', '2024', '2025'])

| Métrica | Estimativa | IC95% inferior | IC95% superior |
| --- | --- | --- | --- |
| accuracy | 68,82% | 68,59% | 69,04% |
| macro_f1 | 67,09% | 66,85% | 67,33% |
| balanced_accuracy | 67,65% | 67,41% | 67,89% |
| recall_roubo | 63,42% | 63,01% | 63,81% |

| Métrica | 2024 | 2025 |
| --- | --- | --- |
| accuracy | 68,51% | 68,82% |
| macro_f1 | 67,69% | 67,09% |
| precision_roubo | 62,58% | 56,15% |
| recall_roubo | 62,53% | 63,42% |
| ece10 | 2,71% | 8,10% |

![Precision, recall e F1 por classe em 2025](../reports/etapa5/desempenho_por_classe.png)

*Precision, recall e F1 por classe em 2025. Figura histórica preservada; não é regenerada nesta consolidação.*

![Scores sigmoid e confiabilidade histórica: não calibrados](../reports/etapa5/avaliacao_probabilidades.png)

*Scores sigmoid e confiabilidade histórica: não calibrados. Figura histórica preservada; não é regenerada nesta consolidação.*

### Generalização e interpretação dos scores

De 2024 para 2025, a prevalência de roubo caiu de 42,07% para 36,22%.
Accuracy subiu 0,31 pp, Macro F1 caiu 0,60 pp e precision de roubo caiu 6,42 pp.
AUC auditada permaneceu próxima (0,7448→0,7412). O ECE10 aumentou de 2,71% para
8,10%; score médio de roubo ≈44,31%, acima da prevalência em ≈8,10 pp.
Há evidência de desajuste de nível associado à mudança de prevalência, sem prova
de invariância de todas as distribuições condicionais.

Brier=0,200547 e Log Loss=0,583506 não isolam calibração. Scores sigmoid são
estimativas não calibradas de classificação, **não chance de crime futuro**.
O Platt candidato de 2024 melhorou medidas probabilísticas, mas foi rejeitado
historicamente pelo recall no limiar 0,5. A errata esclarece que esse critério
confundia calibração com ponto de operação: 0,5 calibrado equivalia a ≈0,5218 na
escala original. A decisão de não adotá-lo e o limiar 0,5 foram preservados.

O diagnóstico de sobreajuste incremental do baseline não deve ser transferido
automaticamente à Dense64 final de três épocas. Não há evidência suficiente de
underfitting de capacidade; contextos com as mesmas entradas têm rótulos distintos.
Mais camadas não mostraram melhora consistente. Erros por local, bairro e mês
são descritivos, com suporte mínimo; local ausente em 2025 tem apenas 68 BOs.

![Variação mensal histórica em 2025, sem seleção posterior](../reports/etapa5/desempenho_mensal_2025.png)

*Variação mensal histórica em 2025, sem seleção posterior. Figura histórica preservada; não é regenerada nesta consolidação.*

### Integridade sem desserializar pesos

A célula seguinte confere o manifesto e oito fontes disponíveis no clone. Se os
binários confiáveis estiverem presentes, confere seus hashes também; sua ausência
é explicitada e não impede a leitura acadêmica. Não abre datasets nem recalcula
resultados. A identidade fixada do manifesto é a aprovada no contrato seguro.

In [16]:
def sha256(arquivo):
    return hashlib.sha256(arquivo.read_bytes()).hexdigest()
hash_manifesto = sha256(ROOT/'models/final/manifesto.json')
assert hash_manifesto == 'b992412335c69623bdf63c9341cefaa82e0a81ae73d7d32e110f2498faa8c532'
assert (ROOT/'models/final/manifesto.sha256').read_text().strip() == hash_manifesto
verificacoes = []
for nome, esperado in {**manifesto['inference_source_hashes'],
                      **{f'models/final/{n}': h for n,h in manifesto['frozen_files'].items()}}.items():
    arquivo = ROOT/nome
    if arquivo.exists():
        assert sha256(arquivo) == esperado, f'Hash divergente: {nome}'
        estado = 'Hash conferido'
    else:
        assert arquivo.suffix in {'.keras', '.joblib'}, f'Fonte pública ausente: {nome}'
        estado = 'Binário local não publicado'
    verificacoes.append({'Arquivo': nome, 'Estado': estado})
tabela(verificacoes, ['Arquivo', 'Estado'])

| Arquivo | Estado |
| --- | --- |
| src/final5/common.py | Hash conferido |
| src/final5/pipeline.py | Hash conferido |
| src/final5/calibration.py | Hash conferido |
| src/experiments4/preprocessing.py | Hash conferido |
| src/experiments4/common.py | Hash conferido |
| src/ml/runtime.py | Hash conferido |
| src/ml/preprocessing.py | Hash conferido |
| src/ml/evaluate.py | Hash conferido |
| models/final/modelo.keras | Hash conferido |
| models/final/preprocessador.joblib | Hash conferido |
| models/final/config_final.json | Hash conferido |
| models/final/contrato.json | Hash conferido |

<a id="08-auditoria"></a>
## 08 — Auditoria independente

Claude realizou a [auditoria técnica](../reports/auditoria_claude.md) em 09/11/2026,
em uma revisão separada do desenvolvimento. Confirmou os números, hashes e pesos;
partiu dos Parquet intermediários, sem rever os XLSX originais, e consultou 2025.
É uma auditoria assistida por IA, não certificação externa ou validação institucional.

| Achado | Natureza e interpretação | Ação |
| --- | --- | --- |
| Referência por frequências | Lacuna de comparação comprovada; tabela retrospectiva aproxima a MLP | Corrigir narrativa; não reivindicar superioridade |
| Rejeição de Platt por recall em 0,5 | Defeito metodológico de interpretação comprovado | Separar calibração de ponto de operação; modelo intacto |
| Ausência/OUTROS em local | Mudança de distribuição comprovada; proxy administrativo plausível | Documentar associação ao canal, sem alegar vazamento comprovado |
| Bairro raro + via pública | 11.118 previsões de roubo, 4.601 FP na auditoria | Limitação da representação compartilhada, não bug da rede |
| Temporal ausente e categoria desconhecida | Defeitos de contrato comprovados | Validação bloqueante externa, antes da rede |
| Normalização fragmentada | Grafias podem produzir categorias distintas | Não mudar o joblib; lista controlada na web |
| Independência/rastreabilidade | Git posterior e teste já consumido | Novo protocolo e novo teste para uma v2 |

Há OUTROS presencial na própria tabela da auditoria em 2025. Portanto, não se
repete sua afirmação absoluta de exclusividade eletrônica: associação ao canal
é sustentada, exclusividade universal não é. Também não se transforma associação
de local/bairro em causalidade criminal ou ranking de segurança.

| Diagnóstico retrospectivo de 2025 | Accuracy | Macro F1 | AUC ROC |
| --- | ---: | ---: | ---: |
| Frequências de 2023 por local/bairro | 69,28% | 67,27% | 0,7389 |
| Dense64 congelada | 68,82% | 67,09% | 0,7412 |

**Não há ganho consistente demonstrado da MLP sobre essa referência.** A tabela
foi construída depois do teste: não é experimento independente, candidata
selecionada ou justificativa para substituir o modelo. AUC mede ordenação ao
longo de limiares; não é Accuracy nem calibração. Os números acima são transcritos
da auditoria; nenhum diagnóstico de 2025 foi recalculado nesta consolidação.

As [correções pós-auditoria](../reports/relatorio_correcoes_pos_auditoria.md)
introduziram `SafeClassifier` (contrato **1.1.0**): mês/dia obrigatórios, domínio
inteiro, categorias reconhecidas ou ausência explícita de local/bairro, rejeição
de extras e desconhecidos, avisos para raros. Pesos, vocabulário, manifesto,
limiar e lógica da rede permaneceram congelados. O pipeline histórico continua
preservado; novas integrações não devem chamá-lo sem o validador seguro.

Hipóteses como regressão logística, encoding geográfico, normalização uniforme,
limiar por custo, recalibração e janelas de treino **não foram implementadas**.
2025 não pode orientar uma v2 apresentada como independente.

<a id="09-web"></a>
## 09 — Integração com a aplicação web

**[Abrir demonstração pública](https://spcrimeclassification.vercel.app/)**

Em 11/11/2026, página, `/api/health`, `/api/options` e `/api/classify` responderam
200. Um exemplo sintético (mês 5, quarta-feira, VIA PÚBLICA, BELA VISTA) retornou
FURTO; scores furto=0,7446408569812775 e roubo=0,25535914301872253, modelo v1.0.0,
contrato 1.1.0. É uma verificação pontual, não promessa de disponibilidade contínua.

```text
Formulário React + TypeScript (Vite)
                ↓ JSON /api/classify
API FastAPI → SafeClassifier (validação bloqueante)
                ↓
Pré-processador congelado → One-Hot 299 colunas
                ↓
Dense64 ReLU → sigmoid → limiar 0,5
                ↓
Classe + scores não calibrados + versões + avisos
```

O frontend usa opções derivadas do vocabulário; não duplica o One-Hot. A API
carrega uma instância por processo, verifica hashes, limita corpo/requisições
e serializa acesso ao Keras. Erros de entrada retornam 422; indisponibilidade
retorna 503, sem resultado fictício. O painel lê agregados públicos pré-computados,
sem Parquet, planilhas ou BOs individuais em execução.

Os binários `.keras` e `.joblib` não estão no GitHub. Para executar localmente,
obtenha os artefatos finais aprovados com o mantenedor e valide o manifesto.
Consulte [execução web](../web/README.md) e o [README principal](../README.md).
Este notebook não dispara requisições de rede automaticamente.

<a id="10-conclusoes"></a>
## 10 — Conclusões

A entrega demonstra preparação auditável de 505.187 BOs, representação categórica
sem atributos jurídicos, três MLPs efetivamente treinadas, comparação temporal,
seleção histórica da Dense64 e integração do artefato congelado a uma aplicação.
Em 2025: **Accuracy 68,82%, Macro F1 67,09%, recall de roubo 63,42%**.

O resultado não mostra que maior profundidade melhora o problema. Dense32 é
competitiva e compacta; Dense64 foi escolhida pela estabilidade examinada e pelo
compromisso de erros. A referência retrospectiva por frequências impede apresentar
a rede como avanço geral demonstrado. A informação contextual disponível é limitada.

Principais restrições: domínio de celulares subtraídos da capital, exclusão
assimétrica de naturezas associadas, grafias de bairros, bucket compartilhado,
possíveis proxies de registro, mudança temporal, scores não calibrados, registros
tardios de 2025 ausentes e intervalos IID que não capturam toda a incerteza.
Nenhuma taxa deve ser usada como risco territorial, orientação policial ou
conclusão causal. Não há finalidade policial operacional validada.

Melhorias futuras exigem novo protocolo de desenvolvimento e teste não consumido
(por exemplo 2026, se disponível e de qualidade adequada). Não alterar esta v1
para acomodar os diagnósticos de 2025. Pesos, pré-processador, limiar e manifesto
permanecem intactos nesta consolidação.

### Uso responsável de IA generativa e autoria

Codex apoiou implementação, documentação, integração e organização desta entrega;
Claude realizou a auditoria técnica registrada. A IA auxiliou código e análise,
mas seus textos exigem conferência contra as evidências: por isso as erratas,
fontes, limitações e resultados históricos permanecem rastreáveis. A responsabilidade
acadêmica é do responsável pela entrega. Não se atribui certificação à auditoria
por IA, nem endosso institucional à FIAP ou à SSP-SP. Nome acadêmico completo,
RM e turma não foram identificados nos documentos acadêmicos consultados;
o repositório público é mantido por [matheeusvx](https://github.com/matheeusvx).

### Referências e aprofundamento

- [FIAP](https://www.fiap.com.br/) · [SSP-SP](https://www.ssp.sp.gov.br/estatistica/consultas)
- [Etapa 1 — fonte e dados](../reports/relatorio_etapa1.md)
- [Etapa 2 — baseline](../reports/relatorio_etapa2.md)
- [Etapa 3 — diagnóstico](../reports/relatorio_etapa3.md)
- [Etapa 4 — experimentos](../reports/relatorio_etapa4.md)
- [Etapa 5 — avaliação final e errata](../reports/relatorio_etapa5.md)
- [Auditoria Claude](../reports/auditoria_claude.md) · [Correções](../reports/relatorio_correcoes_pos_auditoria.md)
- [Limitações](../docs/limitacoes_modelo_final.md) · [Contrato seguro](../docs/contrato_inferencia_seguro.md)
- [Relatório web](../reports/relatorio_etapa6.md) — registro histórico anterior ao deploy público
- [Keras EarlyStopping](https://keras.io/api/callbacks/early_stopping/)
- [scikit-learn: calibração](https://scikit-learn.org/stable/modules/calibration.html)

As declarações de estado dos relatórios refletem suas datas: “sem web” ou “2025
reservado” nas etapas iniciais não descrevem o estado atual. Não foram reescritas.